In [1]:
# 라이브러리 불러오기
from selenium import webdriver as wb # web browser
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time
import pandas as pd
import re
# from tqdm import tqdm
from tqdm.auto import tqdm  # ★ VSCode에선 이게 젤 좋음
from selenium.webdriver.chrome.options import Options
import os
from urllib.parse import quote
from datetime import datetime, timedelta

# 문자열 전처리 함수 -> 숫자, 문자, 특수문자 제외하고 삭제
def preprocess_sentence_kr(w):
  w = w.strip()
  w = re.sub(r"[^0-9가-힣?.!,¿]+", " ", w) 
  w = w.strip() 
  return w

/home/mfdl/anaconda3/envs/py397/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 파일 경로 형식 지정
date = '260303'
ver = 1
output_path = f'./results/티스토리_{date}'

if not os.path.exists(output_path):
    os.makedirs(output_path) # 경로가 없으면 생성 (중간 경로까지 자동 생성)
    print(f"폴더 '{output_path}'를 생성했습니다.")
else:
    print(f"폴더 '{output_path}'는 이미 존재합니다.")

폴더 './results/티스토리_260303'는 이미 존재합니다.


In [3]:
browser_close_flag = True

In [4]:
# 검색어 설정
keywords_lst = [
                # '임신 힘들', 
                # '임신 불안', 
                # '임신 걱정', 
                # '임신 어렵', 
                # '임신 불편',
                # '출산 힘들', 
                # '출산 불안', 
                # '출산 걱정', 
                # '출산 어렵', 
                # '출산 불편',
                # '육아 힘들', 
                # '육아 불안', 
                # '육아 걱정', 
                # '육아 어렵', 
                '육아 불편'
                ]

### Macro

In [5]:
for kidx, keyword in enumerate(keywords_lst):
    titles_list_pd = []
    contents_list_pd = []
    dates_list_pd = []
    href_list_pd = []

    print(f'[keyword {kidx+1}: {keyword}]')

    chrome_options = Options()
    chrome_options.add_experimental_option("detach", True)
    chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
    if(browser_close_flag):
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')

    driver = wb.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30) # 페이지 로딩 타임아웃 설정

    for page_num in range(1, 1000):
        kward = quote(keyword)
        try:
            driver.get(f'https://www.tistory.com/search?keyword={kward}&type=post&sort=ACCURACY&page={page_num}')
        except Exception as e:
            print(f"검색 페이지 로딩 에러: {e}")
            driver.quit()
            driver = wb.Chrome(options=chrome_options) # 브라우저 재시작
            driver.get(f'https://www.tistory.com/search?keyword={kward}&type=post&sort=ACCURACY&page={page_num}')
            
        time.sleep(4.0)

        urls = driver.find_elements(By.CSS_SELECTOR, 'div.list_tistory_top > div.item_group > a')
        if not urls:
            print("[크롤링이 종료되었습니다]")
            break

        href_list = [url.get_attribute('href') for url in urls]
        titles = driver.find_elements(By.CSS_SELECTOR, 'div.wrap_tit > strong')
        titles_list = [title.text for title in titles]
        dates = driver.find_elements(By.CSS_SELECTOR, 'div.wrap_info > span.txt_g')

        # 날짜 처리 (기존 로직 유지)
        dates_list = []
        for date in dates:
            date_ = date.text
            if date_[-3:] == "일 전":
                days_ago = int(date_[:-3])
                calculated_date = datetime.now() - timedelta(days=days_ago)
                dates_list.append(calculated_date.strftime("%Y-%m-%d"))
            else:
                dates_list.append(date_.replace(".", "-"))

        ## 게시글 '내용' 수집 (에러 대응 로직 추가)
        contents_list = []
        for idx, href in enumerate(tqdm(href_list)):
            content = ""
            retry = 0
            while retry < 2: # 에러 시 최대 2회 시도
                try:
                    driver.get(href)
                    time.sleep(1.5)
                    
                    containers = [
                        "div.article-view", "div.tt_article_useless_p_margin",
                        "div.entry-content", "div#content", "article", "main"
                    ]
                    
                    for csel in containers:
                        try:
                            content_area = driver.find_element(By.CSS_SELECTOR, csel)
                            p_tags = content_area.find_elements(By.TAG_NAME, 'p')
                            p_texts = [p.text for p in p_tags if p.text.strip() != ""]
                            if p_texts:
                                content = " ".join(p_texts)
                                break
                        except:
                            continue
                    break # 성공 시 while 탈출
                except Exception as e:
                    print(f"\n[WebDriver Error] 재시도 중... ({retry+1}/2): {e}")
                    driver.quit()
                    time.sleep(2)
                    driver = wb.Chrome(options=chrome_options) # 드라이버 새로 고침
                    retry += 1
            
            if content == "":
                print(f"\n{href} : 본문 내용을 찾지 못했습니다.")
            contents_list.append(content)

        # 리스트 업데이트
        titles_list_pd += titles_list
        contents_list_pd += contents_list
        dates_list_pd += dates_list
        href_list_pd += href_list

    driver.quit()

    ## 데이터 프레임 생성 및 정제 (내용 없는 행 삭제)
    data = {
        '제목': titles_list_pd,
        '내용': contents_list_pd,
        '작성일': dates_list_pd,
        '링크': href_list_pd
    }

    df = pd.DataFrame(data)
    df = df[df['내용'].str.strip() != ""] # 본문 없는 행 제거
    df.insert(0, '출처', '티스토리_%s'%(keyword))
    
    nCnt = len(df)
    df.to_csv("%s/티스토리_%s(%s건).csv"%(output_path, keyword, nCnt), index=False, encoding='utf-8-sig')

[keyword 1: 육아 불편]


 90%|█████████ | 27/30 [01:34<00:09,  3.32s/it]


https://childcare-note.tistory.com/13 : 본문 내용을 찾지 못했습니다.


 93%|█████████▎| 28/30 [01:35<00:05,  2.84s/it]


https://wantchicken.tistory.com/389 : 본문 내용을 찾지 못했습니다.


 97%|█████████▋| 29/30 [01:37<00:02,  2.49s/it]


https://himmel75.tistory.com/entry/%ED%97%AC%EC%8A%A4%EB%B6%80%ED%84%B0-%EC%9C%A1%EC%95%84%EA%B9%8C%EC%A7%80-%EC%86%90%EB%AA%A9-%EB%B6%80%EB%8B%B4%EC%9D%84-%EB%8D%9C%EC%96%B4%EC%A3%BC%EB%8A%94-%EC%95%84%EB%82%98%ED%8C%8C%EC%86%90%EB%AA%A9%EB%B3%B4%ED%98%B8%EB%8C%80 : 본문 내용을 찾지 못했습니다.


100%|██████████| 30/30 [01:39<00:00,  3.31s/it]


https://supermaman.tistory.com/8 : 본문 내용을 찾지 못했습니다.


[크롤링이 종료되었습니다]
